In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append("/home/jg2447/slayman/motif_inference/MOBI/scripts/")
from src import stats
from src import plt_func

### data

In [ ]:
alpha_list_para = np.round(np.append(np.arange(0.1,1.0,0.1), np.arange(1.0,11.0,1.0)), decimals=2)
rank_list_para = ["RankLinear_%.1f" % i for i in alpha_list_para]
rank_list = ["RankSPP"]
rank_list.extend(rank_list_para)
rank_list.extend(["RankCrowdness"])

GM12878_result = []
K562_result = []
for rank in rank_list:
    spp = pd.read_csv("/home/jg2447/slayman/motif_inference/result/CellLineColdnessFromNoGK/GM12878/tomtom_summary/DREME_RankSPP_100_top5.txt", sep="\t")
    df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/CellLineColdnessFromNoGK/GM12878/tomtom_summary/DREME_%s_100_top5.txt" % rank, sep="\t")
    test_p = stats.sig_test_wilcoxon(df['accuracy'], spp['accuracy'])
    if test_p < 0.05:
        p = "*"
    else:
        p = ""
    improve = (df['accuracy'].mean() - spp['accuracy'].mean())/spp['accuracy'].mean()
    GM12878_result.append((improve, p))
    
    spp = pd.read_csv("/home/jg2447/slayman/motif_inference/result/CellLineColdnessFromNoGK/K562/tomtom_summary/DREME_RankSPP_100_top5.txt", sep="\t")
    df = pd.read_csv("/home/jg2447/slayman/motif_inference/result/CellLineColdnessFromNoGK/K562/tomtom_summary/DREME_%s_100_top5.txt" % rank, sep="\t")
    test_p = stats.sig_test_wilcoxon(df['accuracy'], spp['accuracy'])
    if test_p < 0.05:
        p = "*"
    else:
        p = ""
    improve = (df['accuracy'].mean() - spp['accuracy'].mean())/spp['accuracy'].mean()
    K562_result.append((improve, p))

GM12878_result = pd.DataFrame(GM12878_result)
GM12878_result.columns = ["accuracy", "p"]
GM12878_result.to_csv("./figS1_data_GM12878.txt", sep="\t", index=False)

K562_result = pd.DataFrame(K562_result)
K562_result.columns = ["accuracy", "p"]
K562_result.to_csv("./figS1_data_K562.txt", sep="\t", index=False)

### plot

In [ ]:
def plt_heatmap_rank(data, data_p, axs, vmax):
    # axs: list of ax: 0 - spp, 1 - BC, 2 - C, 3 - cbar
    
    # SPP part
    ax1 = sns.heatmap(
        pd.DataFrame([data]).iloc[:, 0:1],
        ax=axs[0],
        annot=pd.DataFrame([data_p]).iloc[:, 0:1],
        cbar=False,
        vmax=vmax, vmin=-vmax, center=0, cmap=cmap, annot_kws={"fontsize":6}, fmt="s", linewidths=.05)
    # BC-score part
    ax2 = sns.heatmap(
        pd.DataFrame([data]).iloc[:, 1:-1],
        ax=axs[1],
        annot=pd.DataFrame([data_p]).iloc[:, 1:-1],
        cbar_ax=axs[3], cbar_kws={"orientation": "horizontal"}, # add color bar
        vmax=vmax, vmin=-vmax, center=0, cmap=cmap, annot_kws={"fontsize":6}, fmt="s", linewidths=.05)
    # C-score part
    ax3 = sns.heatmap(
        pd.DataFrame([data]).iloc[:, -1:],
        ax=axs[2],
        annot=pd.DataFrame([data_p]).iloc[:, -1:],
        cbar=False,
        vmax=vmax, vmin=-vmax, center=0, cmap=cmap, annot_kws={"fontsize":6}, fmt="s", linewidths=.05)
       
    # no tick/label
    ax1 = plt_func.rm_axis_label(ax1, "y")    
    ax2 = plt_func.rm_axis_label(ax2, "y")    
    ax3 = plt_func.rm_axis_label(ax3, "y")
    ax1 = plt_func.rm_axis_label(ax1, "x")
    ax2 = plt_func.rm_axis_label(ax2, "x")
    ax3 = plt_func.rm_axis_label(ax3, "x")
    ax1.yaxis.set_ticks([0.5])
    
    return((ax1, ax2, ax3))

In [ ]:
figsize = (3.2, 0.85)

panel_number_fs = 8
x_tick_label_fs = 6
y_tick_label_fs = 6
x_label_fs = 7
y_label_fs = 7
title_fs = 7
legend_fs = 5

color = ["#c2d2ed", "#ed6a6d", "#c2d2ed", "#ed6a6d"]
cmap = "RdBu"

sns.set_context("paper")


fig = plt.figure(figsize=figsize, dpi=300)
gs = mpl.gridspec.GridSpec(
    5, 5, 
    figure=fig,
    hspace=0.2,
    wspace=0.05,
    height_ratios=[0.3, 1, 0.1, 1, 2.2],
    width_ratios=[1,6,7,6,1])

ax_cbar = fig.add_subplot(gs[0,2])
vmax = np.max([GM12878_result['accuracy'].abs().max(), K562_result['accuracy'].abs().max()])
##
ax1 = fig.add_subplot(gs[1,0]) # spp
ax2 = fig.add_subplot(gs[1,1:4]) # bc-score
ax3 = fig.add_subplot(gs[1,4]) # c-score

ax1, ax2, ax3 = plt_heatmap_rank(GM12878_result['accuracy'].values, GM12878_result['p'].values, (ax1, ax2, ax3, ax_cbar), vmax)
ax1.yaxis.set_ticklabels(["GM12878"], rotation=0, fontsize=y_tick_label_fs)
ax1.annotate("★", (0.15, 0.75), fontsize=6, color='red')

## 
ax1 = fig.add_subplot(gs[3,0])
ax2 = fig.add_subplot(gs[3,1:4])
ax3 = fig.add_subplot(gs[3,4])
ax1, ax2, ax3 = plt_heatmap_rank(K562_result['accuracy'].values, K562_result['p'].values, (ax1, ax2, ax3, ax_cbar), vmax)
ax1.yaxis.set_ticklabels(["K562"], rotation=0, fontsize=y_tick_label_fs)
ax1.annotate("★", (0.15, 0.75), fontsize=6, color='red')

# no color bar tick/label
ax_cbar = plt_func.rm_axis_label(ax_cbar, "x")
# color bar label
ax_cbar.annotate("%.2f" % (-vmax), xy=(-15,0.2), fontsize=legend_fs, xycoords="axes pixels")
ax_cbar.annotate("%.2f" % (vmax), xy=(59,0.2), fontsize=legend_fs, xycoords="axes pixels")

# X ticks and labels
ax1.xaxis.set_ticks([0.5])
ax1.xaxis.set_ticklabels(["SPP"], fontsize=x_tick_label_fs-1, rotation=90)

ax2.xaxis.set_ticks(np.arange(0.5,19.5,1))
ax2.xaxis.set_ticklabels(
    ["0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "1.0", "2.0", "3.0", "4.0", "5.0", "6.0", "7.0", "8.0", "9.0", "10.0"],
    fontsize=x_tick_label_fs-1, rotation=90)

ax3.xaxis.set_ticks([0.5])
ax3.xaxis.set_ticklabels(["C-score"], fontsize=x_tick_label_fs-1, rotation=90)

for axx in [ax1, ax2, ax3]:
    axx.xaxis.set_tick_params(size=2.5, pad=1, color="#757575")

ax2.set_xlabel("Binding site ranking method", fontsize=x_label_fs, labelpad=7)

# BC-score annotation
ax_annot = fig.add_subplot(gs[4,1:4])
ax_annot.plot([1,21], [-1,-1], ",", color="white")
ax_annot.set_ylim([0,2])
ax_annot.set_xlim([0,21])    
ax_annot.annotate("", (0.5, 0.4), (5.5, 0.4), arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", (15.5, 0.4), (20.5, 0.4), arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", (0.5, 0), (0.5, 0.8), arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", (20.5, 0), (20.5, 0.8), arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("BC-score with different weight", (5.5, 0.25), (5.5, 0.25), rotation=0, size=5)
plt_func.rm_axis_label(ax_annot, "x")
ax_annot.axis('off')

plt.savefig("./figS1.pdf", dpi='figure', bbox_inches="tight")